# RAG System - Document Q&A & Chatbot

This notebook provides a complete RAG system with:
- Hybrid retrieval (BGE + BM25 + FAISS)
- Ollama/gemma4 generation
- Simple query interface
- Interactive chatbot

In [1]:
# Setup and Imports
import sys
import os
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path('.').absolute()))

from src.rag_system import RAGSystem
print('RAG System imported successfully')

W0521 16:43:15.407000 42884 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.



RAG System imported successfully


## Initialize RAG System

In [2]:
# Initialize RAG system with source directory
SOURCE_DIR = Path('sources')

rag = RAGSystem(source_dir=SOURCE_DIR)
print('RAG System initialized')

# Check Ollama and preload model
from src.ollama_manager import OllamaManager

print("\n=== Ollama Status Check ===")
ollama_mgr = OllamaManager(
    base_url=rag.generator.config.base_url,
    model=rag.generator.config.model
)

status = ollama_mgr.get_status_summary()
if status["ollama_available"]:
    print("✅ Ollama is running")
    print(f"Available models: {status['available_models']}")

    if status["matching_model"]:
        print(f"✅ Model '{status['matching_model']}' is available")
        print(f"\n⏳ Loading model into memory...")
        success, elapsed = ollama_mgr.preload_model(status["matching_model"])
        if success:
            print(f"✅ Model ready ({elapsed:.1f}s)")
            rag.generator.config.model = status["matching_model"]
        else:
            print(f"⚠️ Model may not be fully loaded")
    else:
        print(f"⚠️ Configured model '{status['configured_model']}' not found")
else:
    print("❌ Ollama is NOT running")
    print("   Start with: `ollama serve`")

RAG System initialized

=== Ollama Status Check ===
✅ Ollama is running
Available models: ['nomic-embed-text:latest', 'gemma4:e4b', 'qwen3.5:4b', 'mistrallite:7b', 'mistral-openorca:7b', 'mistral:7b', 'qwen2:7b', 'qwen3:8b', 'qwen2.5:7b', 'llama3:8b', 'llama3.2:3b', 'gemma3:4b', 'gemma:7b', 'gemma2:9b', 'gemini-3-flash-preview:cloud', 'gemini-3-flash-preview:latest', 'llama3.1:8b', 'mistral-nemo:12b']
✅ Model 'gemma4:e4b' is available

⏳ Loading model into memory...
✅ Model ready (36.7s)


## Ingest Documents

In [3]:
# Ingest and index all documents
stats = rag.ingest_documents(SOURCE_DIR)
print(f"Ingested {stats['documents_loaded']} documents")
print(f"Created {stats['chunks_created']} chunks")
print(f"System stats: {rag.get_stats()}")

Loading cached index for 6 files...
Loading vector index from .rag_index\vectors.faiss...
Loading BM25 index from .rag_index\bm25.json...
Loaded 813 chunks from .rag_index\chunks.json
Ingested 6 documents
Created 810 chunks
System stats: {'indexed': True, 'document_count': 813, 'model': 'gemma4:e4b', 'embedding_model': 'BAAI/bge-large-en-v1.5', 'index_dir': '.rag_index'}


## Example Queries

In [4]:
# # Function to query the RAG system
# def ask(question: str, top_k: int = 3):
#     """Ask a question and display results."""
#     result = rag.query(question, top_k=top_k)
    
#     print(f"Question: {question}\n")
#     print(f"Answer: {result.answer}\n")
#     print("Sources:")
#     for i, source in enumerate(result.sources, 1):
#         print(f"  [{i}] {source['source']}, Page {source['page']} (score: {source['score']:.3f})")
    
#     return result

In [5]:
# Example: Ask about security
# ask("What are the main security concerns for AI agents?")

In [6]:
# Example: Ask about OWASP Top 10
# ask("What does the OWASP Top 10 say about AI security?")

## Interactive Chat

In [7]:
# Interactive chat function
# def chat():
#     """Interactive chat loop."""
#     print('RAG Chatbot - Type your question or "quit" to exit\n')
    
#     while True:
#         question = input('You: ')
#         if question.lower() in ['quit', 'exit', 'q']:
#             print('Goodbye!')
#             break
        
#         result = rag.query(question)
#         print(f"\nAssistant: {result.answer}\n")
#         print('-' * 50)

In [8]:
# Start interactive chat (uncomment to use)
# chat()

In [9]:
# RAGAS Evaluation
from src.test_case import CASES
from src.evaluator import RAGASEvaluator

# Initialize evaluator with RAG system
evaluator = RAGASEvaluator(rag)
print("Evaluator initialized")

c:\Users\sam99\AppData\Local\Programs\Python\Python311\Lib\site-packages\instructor\providers\gemini\client.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai  # type: ignore[import-not-found]


Evaluator initialized


## Evaluation

In [10]:
# Configure evaluation settings
from src.config import config

print("Evaluation configuration:")
print(f"  Model: {config.eval_llm.model}")
print(f"  Base URL: {config.eval_llm.base_url}")
print(f"  Metrics: {evaluator.metrics}")

Evaluation configuration:
  Model: gemma4:e4b
  Base URL: http://localhost:11434/v1
  Metrics: ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']


In [11]:
# Run evaluation on all test cases
results = evaluator.run_batch(CASES)
print(f"Evaluated {len(results)} test cases")

Evaluated 10 test cases


In [12]:
# Display evaluation results
print(f"Running RAGAS evaluation on {len(CASES)} test cases...")

evaluator.print_results(results)
evaluator.save_results(results)

Running RAGAS evaluation on 10 test cases...

RAGAS EVALUATION RESULTS
Question                                 Faith    Relev    Prec     Recall  
--------------------------------------------------------------------------------
What is prompt injection in LLM appli... 1.00    0.56    0.00    1.00
  [Answer 1]: Indirect Prompt Injection involves several risks, including hidden instruction payloads embedded in ...
  [1] OWASP-Top-10-for-Agentic-Applications-2026-12.6-1.pdf, Page 10 (score: 0.032)
  [2] AUTOATTACKER A Large Language Model Guided System to Implement Automatic Cyber-attacks.pdf, Page 7 (score: 0.031)
  [3] AUTOATTACKER A Large Language Model Guided System to Implement Automatic Cyber-attacks.pdf, Page 6 (score: 0.029)
What does the OWASP Top 10 for LLM sa... 1.00    0.65    0.00    0.00
  [Answer 2]: The context suggests that protecting memory from data leakage is part of planning for the agent's sh...
  [1] OWASP-Top-10-for-Agentic-Applications-2026-12.6-1.pdf, Page 7 (sc

## Evaluation Configuration